# Training and Evaluation in one Notebook for One Model-Database Pair

# To check before running
1. Check class names for your event log in the **p2pencoder.py** ( *{event_log_name}encoder.py* )
2. Check the Axioms in **axiombuilder.py**
3. make sure you have done the declare mining on the event log and have a valid **ltn_rows_path**

In [64]:
event_log_name = "bpic17"
if event_log_name is None:
    raise ValueError("Please set the event_log_name variable to the name of the event log you want to use.")
ltn_rows_path = f"{event_log_name}_ltn_rows.pkl"
print(f"Event log name {event_log_name}")
print(f"LTN Rows path {ltn_rows_path}")
# starting time


Event log name bpic17
LTN Rows path bpic17_ltn_rows.pkl


In [65]:
# import tensorflow as tf
# physical_devices = tf.config.list_physical_devices('GPU')
# print(physical_devices)
# if len(physical_devices) > 0:
#     tf.config.experimental.set_memory_growth(physical_devices[0], True)
#     print("GPU found")
#     print("Memory growth set")
# else:
#     print("No GPU found")

In [66]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

import itertools

from sklearn import metrics


from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.bpic17evaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

import matplotlib.pyplot as plt
import numpy as np
np.random.seed(0)

import pandas as pd
import seaborn as sns
from sqlalchemy.orm import Session
import scikit_posthocs as sp

from april.database import get_engine
from april.fs import PLOT_DIR
from april.utils import microsoft_colors, prettify_dataframe, cd_plot, get_cd
from april.enums import Base, Strategy, Heuristic

sns.set_style('white')
pd.set_option('display.max_rows', 50)
%config InlineBackend.figure_format = 'retina'
print(bpic17_leaky_row_classes)

[<class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-10'>, <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-25'>, <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-50'>, <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-100'>, <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-150'>, <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-200'>, <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-250'>, <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-300'>]


In [67]:
dataset = f"{event_log_name}-0.3-1"
out_dir = PLOT_DIR / f'{event_log_name}_evaluations_both_{arrow.now().format("YYYY-MM-DD-HH-mm-ss")}'
eval_file = out_dir / f'{event_log_name}_fraction_evaluations.pkl'
csv_file = out_dir / f'{event_log_name}_fraction_evaluations.csv'
excel_file = out_dir / f'{event_log_name}_fraction_evaluations.xlsx'
model_folder = r"D:\LTNcoder\.out\models"
db = r"D:\LTNcoder\.out\april.db"

# create out_dir if it does not exist
if not out_dir.exists():
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {out_dir}")
from april.utils import delete_all_files_in_folder, delete_evaluation_and_model_tables
delete_all_files_in_folder(model_folder)
delete_evaluation_and_model_tables(db)
start_time = arrow.now("Europe/Berlin")


Created directory: d:\LTNcoder\.out\plots\bpic17_evaluations_both_2025-08-17-10-31-54
Deleted all rows from Evaluation and Model tables.


# Training

In [68]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    dataset = Dataset(dataset_name)

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()
    pass

In [69]:
ads = [
        dict(ad=Bpic17DAE, fit_kwargs=dict(epochs=6, batch_size=100)),
    ] + \
    [
        dict(ad=LEAKY_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100)) for LEAKY_ROW_CLASS 
        in bpic17_leaky_row_classes
    ] + \
    [
        dict(ad=LTN_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100, epochs_ltn=3))
        for LTN_ROW_CLASS in bpic17_ltn_row_classes
    ]
print(ads)
for ad in tqdm(ads, desc="Fitting ADs"):
    fit_and_save(dataset, **ad)


[{'ad': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-10'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-25'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-50'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-100'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-150'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-200'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-250'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april

Fitting ADs:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 1/6
54/54 [==============================] - 1s 14ms/step - loss: 0.2219 - accuracy: 0.0554 - val_loss: 0.1599 - val_accuracy: 0.0185
Epoch 2/6
54/54 [==============================] - 0s 8ms/step - loss: 0.0573 - accuracy: 0.3398 - val_loss: 0.0121 - val_accuracy: 0.0000e+00
Epoch 3/6
54/54 [==============================] - 0s 7ms/step - loss: 0.0137 - accuracy: 0.3991 - val_loss: 0.0114 - val_accuracy: 0.0000e+00
Epoch 4/6
54/54 [==============================] - 0s 7ms/step - loss: 0.0127 - accuracy: 0.4107 - val_loss: 0.0111 - val_accuracy: 0.0000e+00
Epoch 5/6
54/54 [==============================] - 0s 5ms/step - loss: 0.0123 - accuracy: 0.4464 - val_loss: 0.0107 - val_accuracy: 0.0084
Epoch 6/6
54/54 [==============================] - 0s 5ms/step - loss: 0.0118 - accuracy: 0.4718 - val_loss: 0.0099 - val_accuracy: 0.4141
d:\LTNcoder\.out\models\bpic17-0.3-1_bpic17dae_20250817-103154.762177.keras
Loading model bpic17-0.3-1_bpic17dae_20250817-103154.762177 / <april.fs.Model

In [70]:
print(AD) #Evaluator dependso on AD

{'binetv0': <class 'april.anomalydetection.binet.binet.BINetv0'>, 'binetv1': <class 'april.anomalydetection.binet.binet.BINetv1'>, 'binetv2': <class 'april.anomalydetection.binet.binet.BINetv2'>, 'binetv3': <class 'april.anomalydetection.binet.binet.BINetv3'>, 'likelihood': <class 'april.anomalydetection.boehmer.BoehmerLikelihoodAnomalyDetector'>, 'bpic17dae': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE'>, 'bpic17dae-leaky-10': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-10'>, 'bpic17dae-leaky-100': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-100'>, 'bpic17dae-leaky-150': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-150'>, 'bpic17dae-leaky-200': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-200'>, 'bpic17dae-leaky-25': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-25'>, 'bpic17dae-leaky-250': <class 'april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-250'>, 'bpic17dae-leaky-300': <class 'april.

# Evaluation

In [71]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [72]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    print(f"{e} loaded.")
    # print attributes of e
    print(f"e.model_file: {e.model_file}")
    print(f"e.model_name: {e.model_name}")
    print(f"e.eventlog_name: {e.eventlog_name}")
    print(f"e.dataset: {e.dataset}")
    print(f"e.result: {e.result}")
    
    
    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        # print(f"Adding parameters: {e}, {base}, {heuristic}, {strategy}")
        _params.append([e, base, heuristic, strategy])
    
    print(f"{_params} parameters to evaluate.")

    return [_e for p in _params for _e in _evaluate(p)]

In [73]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(f"Available Models: {models}")
evaluations = []
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

Available Models: ['bpic17-0.3-1_bpic17dae-leaky-100_20250817-103207.695380', 'bpic17-0.3-1_bpic17dae-leaky-10_20250817-103158.392443', 'bpic17-0.3-1_bpic17dae-leaky-150_20250817-103210.568950', 'bpic17-0.3-1_bpic17dae-leaky-200_20250817-103214.121128', 'bpic17-0.3-1_bpic17dae-leaky-250_20250817-103217.874887', 'bpic17-0.3-1_bpic17dae-leaky-25_20250817-103201.545182', 'bpic17-0.3-1_bpic17dae-leaky-300_20250817-103220.837186', 'bpic17-0.3-1_bpic17dae-leaky-50_20250817-103204.548319', 'bpic17-0.3-1_bpic17dae_20250817-103154.762177', 'bpic17-0.3-1_bpic17ltnfrozen-100_20250817-103351.957225', 'bpic17-0.3-1_bpic17ltnfrozen-10_20250817-103223.946201', 'bpic17-0.3-1_bpic17ltnfrozen-150_20250817-103421.597890', 'bpic17-0.3-1_bpic17ltnfrozen-200_20250817-103509.512337', 'bpic17-0.3-1_bpic17ltnfrozen-250_20250817-103558.511496', 'bpic17-0.3-1_bpic17ltnfrozen-25_20250817-103253.009910', 'bpic17-0.3-1_bpic17ltnfrozen-300_20250817-103644.872668', 'bpic17-0.3-1_bpic17ltnfrozen-50_20250817-103322.437

Evaluate:   0%|          | 0/17 [00:00<?, ?it/s]

Evaluating bpic17-0.3-1_bpic17dae-leaky-100_20250817-103207.695380...
Loading model bpic17-0.3-1_bpic17dae-leaky-100_20250817-103207.695380 / <april.fs.ModelFile object at 0x0000017D019BE130> for event log bpic17-0.3-1 at path d:\LTNcoder\.out\models\bpic17-0.3-1_bpic17dae-leaky-100_20250817-103207.695380.keras
Self.ad_: <april.anomalydetection.bpic17encoder.Bpic17DAE-Leaky-100 object at 0x0000017D019BE400>
<april.bpic17evaluator.Evaluator object at 0x0000017D019BED60> loaded.
e.model_file: d:\LTNcoder\.out\models\bpic17-0.3-1_bpic17dae-leaky-100_20250817-103207.695380.keras
e.model_name: bpic17-0.3-1_bpic17dae-leaky-100_20250817-103207.695380
e.eventlog_name: bpic17-0.3-1
Filtering dataset to 531 LTN rows.
Indices: [15, 19, 20, 21, 29, 36, 37, 39, 59, 90, 101, 129, 136, 165, 179, 181, 192, 201, 213, 217, 230, 238, 239, 254, 255, 268, 276, 313, 315, 318, 326, 347, 366, 374, 391, 398, 405, 425, 432, 445, 447, 450, 460, 490, 494, 502, 511, 549, 565, 580, 598, 613, 628, 632, 644, 684, 686

In [74]:

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

  0%|          | 0/1224 [00:00<?, ?it/s]

In [75]:
synth_datasets = ['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide']
bpic_datasets = ['bpic12', 'bpic13', 'bpic15', 'bpic17']
anonymous_datasets = ['real']
datasets = synth_datasets + bpic_datasets + anonymous_datasets
dataset_types = ['Synthetic', 'Real-life']

orig_ads = [ad['ad'].__name__ for ad in ads if "DAE" in ad['ad'].__name__]
new_ads = [ad['ad'].__name__ for ad in ads if "DAE" not in ad['ad'].__name__]
ads = orig_ads + new_ads

heuristics = [r'$best$', r'$default$', r'$elbow_\downarrow$', r'$elbow_\uparrow$', 
              r'$lp_\leftarrow$', r'$lp_\leftrightarrow$', r'$lp_\rightarrow$']
print(ads)

['Bpic17DAE', 'Bpic17DAE-Leaky-10', 'Bpic17DAE-Leaky-25', 'Bpic17DAE-Leaky-50', 'Bpic17DAE-Leaky-100', 'Bpic17DAE-Leaky-150', 'Bpic17DAE-Leaky-200', 'Bpic17DAE-Leaky-250', 'Bpic17DAE-Leaky-300', 'Bpic17LTNFROZEN-10', 'Bpic17LTNFROZEN-25', 'Bpic17LTNFROZEN-50', 'Bpic17LTNFROZEN-100', 'Bpic17LTNFROZEN-150', 'Bpic17LTNFROZEN-200', 'Bpic17LTNFROZEN-250', 'Bpic17LTNFROZEN-300']


In [76]:
evaluation = evaluation.query(f'ad in {ads} and label == "Anomaly"')
display(evaluation)

,file_name,date,hyperparameters,training_duration,training_host,ad,dataset_name,process_model,noise,dataset_id,axis,base,heuristic,strategy,label,attribute_name,perspective,precision,recall,f1
1,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,0,scores,best,single,Anomaly,name,Control Flow,0.494505,0.681818,0.573248
3,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,1,scores,best,single,Anomaly,name,Control Flow,0.138643,0.186508,0.159052
5,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,2,scores,best,single,Anomaly,name,Control Flow,0.138643,0.186508,0.159052
7,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,0,scores,best,position,Anomaly,name,Control Flow,0.248588,1.000000,0.398190
9,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,1,scores,best,position,Anomaly,name,Control Flow,0.068114,0.361111,0.114610
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1215,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,1,scores,stable_right,single,Anomaly,name,Control Flow,0.714286,0.019841,0.038610
1217,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,2,scores,stable_right,single,Anomaly,name,Control Flow,0.714286,0.019841,0.038610
1219,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,0,scores,stable_right,position,Anomaly,name,Control Flow,0.349854,0.909091,0.505263
1221,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,1,scores,stable_right,position,Anomaly,name,Control Flow,0.072727,0.222222,0.109589


In [77]:
evaluation['perspective-label'] = evaluation['perspective'] + '-' + evaluation['label']
evaluation['attribute_name-label'] = evaluation['attribute_name'] + '-' + evaluation['label']
evaluation['dataset_type'] = 'Synthetic'
evaluation.loc[evaluation['process_model'].str.contains('bpic'), 'dataset_type'] = 'Real-life'
evaluation.loc[evaluation['process_model'].str.contains('real'), 'dataset_type'] = 'Real-life'

In [78]:
_filtered_evaluation = evaluation.query(f'ad in {ads} and (strategy == "{Strategy.ATTRIBUTE}" or strategy == "{Strategy.POSITION}"'
                                       f' or (strategy == "{Strategy.SINGLE}" and process_model == "bpic12")'
                                       f' or (strategy == "{Strategy.SINGLE}" and ad == "Naive+"))')

In [79]:
display(_filtered_evaluation)

,file_name,date,hyperparameters,training_duration,training_host,ad,dataset_name,process_model,noise,dataset_id,...,strategy,label,attribute_name,perspective,precision,recall,f1,perspective-label,attribute_name-label,dataset_type
7,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.248588,1.000000,0.398190,Control Flow-Anomaly,name-Anomaly,Real-life
9,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.068114,0.361111,0.114610,Control Flow-Anomaly,name-Anomaly,Real-life
11,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.068114,0.361111,0.114610,Control Flow-Anomaly,name-Anomaly,Real-life
19,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.248588,1.000000,0.398190,Control Flow-Anomaly,name-Anomaly,Real-life
21,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.035544,0.797619,0.068055,Control Flow-Anomaly,name-Anomaly,Real-life
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1209,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.080194,0.261905,0.122791,Control Flow-Anomaly,name-Anomaly,Real-life
1211,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.080194,0.261905,0.122791,Control Flow-Anomaly,name-Anomaly,Real-life
1219,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.349854,0.909091,0.505263,Control Flow-Anomaly,name-Anomaly,Real-life
1221,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.072727,0.222222,0.109589,Control Flow-Anomaly,name-Anomaly,Real-life


In [80]:
filtered_evaluation = _filtered_evaluation.query(f'heuristic == "{Heuristic.DEFAULT}"'
                                                 f' or (heuristic == "{Heuristic.LP_MEAN}" and ad in {orig_ads})'
                                                 f' or (heuristic == "{Heuristic.LP_LEFT}" and ad in {new_ads})'
                                                )

In [81]:
display(filtered_evaluation)

,file_name,date,hyperparameters,training_duration,training_host,ad,dataset_name,process_model,noise,dataset_id,...,strategy,label,attribute_name,perspective,precision,recall,f1,perspective-label,attribute_name-label,dataset_type
55,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.440000,0.583333,0.501629,Control Flow-Anomaly,name-Anomaly,Real-life
57,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.116208,0.150794,0.131261,Control Flow-Anomaly,name-Anomaly,Real-life
59,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.116208,0.150794,0.131261,Control Flow-Anomaly,name-Anomaly,Real-life
127,bpic17-0.3-1_bpic17dae-leaky-10_20250817-10315...,2025-08-17 10:32:01.223929,"{'epochs': 6, 'batch_size': 100}",2.831486,Dev-RTX,Bpic17DAE-Leaky-10,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.439560,0.606061,0.509554,Control Flow-Anomaly,name-Anomaly,Real-life
129,bpic17-0.3-1_bpic17dae-leaky-10_20250817-10315...,2025-08-17 10:32:01.223929,"{'epochs': 6, 'batch_size': 100}",2.831486,Dev-RTX,Bpic17DAE-Leaky-10,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.160142,0.178571,0.168856,Control Flow-Anomaly,name-Anomaly,Real-life
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1125,bpic17-0.3-1_bpic17ltnfrozen-300_20250817-1036...,2025-08-17 10:37:35.211201,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",50.338533,Dev-RTX,Bpic17LTNFROZEN-300,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.103060,0.253968,0.146621,Control Flow-Anomaly,name-Anomaly,Real-life
1127,bpic17-0.3-1_bpic17ltnfrozen-300_20250817-1036...,2025-08-17 10:37:35.211201,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",50.338533,Dev-RTX,Bpic17LTNFROZEN-300,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.103060,0.253968,0.146621,Control Flow-Anomaly,name-Anomaly,Real-life
1195,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.339726,0.939394,0.498994,Control Flow-Anomaly,name-Anomaly,Real-life
1197,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.088172,0.325397,0.138748,Control Flow-Anomaly,name-Anomaly,Real-life


In [ ]:
df = filtered_evaluation.query('axis == 0')
display(filtered_evaluation)
df = prettify_dataframe(df)
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name', 'perspective'])[['precision', 'recall', 'f1']].mean().reset_index()
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name'])[['precision', 'recall', 'f1']].mean().reset_index()
df['f1'] = 2 * df['recall'] * df['precision'] / (df['recall'] + df['precision'])

df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model', 'dataset_name'], values=['precision', 'recall', 'f1'])
df = df.fillna(0)
df = df.stack(1).stack(1).reset_index()
df.to_excel(str(out_dir / 'table.xlsx'), index=False)

# drop rows in column "axis" which have value "Attribute"
df = df.query('axis != "Attribute"')

# df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model'], values=['precision', 'recall', 'f1'], aggfunc=np.mean)

df.to_excel(str(excel_file), index=False)
df.to_csv(str(csv_file), index=False)
print(df)

,file_name,date,hyperparameters,training_duration,training_host,ad,dataset_name,process_model,noise,dataset_id,...,strategy,label,attribute_name,perspective,precision,recall,f1,perspective-label,attribute_name-label,dataset_type
55,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.440000,0.583333,0.501629,Control Flow-Anomaly,name-Anomaly,Real-life
57,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.116208,0.150794,0.131261,Control Flow-Anomaly,name-Anomaly,Real-life
59,bpic17-0.3-1_bpic17dae-leaky-100_20250817-1032...,2025-08-17 10:32:10.272998,"{'epochs': 6, 'batch_size': 100}",2.577618,Dev-RTX,Bpic17DAE-Leaky-100,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.116208,0.150794,0.131261,Control Flow-Anomaly,name-Anomaly,Real-life
127,bpic17-0.3-1_bpic17dae-leaky-10_20250817-10315...,2025-08-17 10:32:01.223929,"{'epochs': 6, 'batch_size': 100}",2.831486,Dev-RTX,Bpic17DAE-Leaky-10,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.439560,0.606061,0.509554,Control Flow-Anomaly,name-Anomaly,Real-life
129,bpic17-0.3-1_bpic17dae-leaky-10_20250817-10315...,2025-08-17 10:32:01.223929,"{'epochs': 6, 'batch_size': 100}",2.831486,Dev-RTX,Bpic17DAE-Leaky-10,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.160142,0.178571,0.168856,Control Flow-Anomaly,name-Anomaly,Real-life
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1125,bpic17-0.3-1_bpic17ltnfrozen-300_20250817-1036...,2025-08-17 10:37:35.211201,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",50.338533,Dev-RTX,Bpic17LTNFROZEN-300,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.103060,0.253968,0.146621,Control Flow-Anomaly,name-Anomaly,Real-life
1127,bpic17-0.3-1_bpic17ltnfrozen-300_20250817-1036...,2025-08-17 10:37:35.211201,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",50.338533,Dev-RTX,Bpic17LTNFROZEN-300,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.103060,0.253968,0.146621,Control Flow-Anomaly,name-Anomaly,Real-life
1195,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.339726,0.939394,0.498994,Control Flow-Anomaly,name-Anomaly,Real-life
1197,bpic17-0.3-1_bpic17ltnfrozen-50_20250817-10332...,2025-08-17 10:33:51.664030,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",29.226254,Dev-RTX,Bpic17LTNFROZEN-50,bpic17-0.3-1,bpic17,0.3,1,...,position,Anomaly,name,Control Flow,0.088172,0.325397,0.138748,Control Flow-Anomaly,name-Anomaly,Real-life


    axis                   ad process_model  dataset_name        f1  \
0   Case            Bpic17DAE        BPIC17  bpic17-0.3-1  0.517350   
1   Case   Bpic17DAE-Leaky-10        BPIC17  bpic17-0.3-1  0.509554   
2   Case  Bpic17DAE-Leaky-100        BPIC17  bpic17-0.3-1  0.501629   
3   Case  Bpic17DAE-Leaky-150        BPIC17  bpic17-0.3-1  0.544218   
4   Case  Bpic17DAE-Leaky-200        BPIC17  bpic17-0.3-1  0.511111   
5   Case   Bpic17DAE-Leaky-25        BPIC17  bpic17-0.3-1  0.492424   
6   Case  Bpic17DAE-Leaky-250        BPIC17  bpic17-0.3-1  0.546713   
7   Case  Bpic17DAE-Leaky-300        BPIC17  bpic17-0.3-1  0.515901   
8   Case   Bpic17DAE-Leaky-50        BPIC17  bpic17-0.3-1  0.480818   
9   Case   Bpic17LTNFROZEN-10        BPIC17  bpic17-0.3-1  0.521505   
10  Case  Bpic17LTNFROZEN-100        BPIC17  bpic17-0.3-1  0.492099   
11  Case  Bpic17LTNFROZEN-150        BPIC17  bpic17-0.3-1  0.486275   
12  Case  Bpic17LTNFROZEN-200        BPIC17  bpic17-0.3-1  0.494024   
13  Ca

C:\Users\devas\AppData\Local\Temp\ipykernel_21344\3173554955.py:10: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()
C:\Users\devas\AppData\Local\Temp\ipykernel_21344\3173554955.py:10: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()


In [83]:
display(df)

,axis,ad,process_model,dataset_name,f1,precision,recall
0,Case,Bpic17DAE,BPIC17,bpic17-0.3-1,0.517350,0.443243,0.621212
1,Case,Bpic17DAE-Leaky-10,BPIC17,bpic17-0.3-1,0.509554,0.439560,0.606061
2,Case,Bpic17DAE-Leaky-100,BPIC17,bpic17-0.3-1,0.501629,0.440000,0.583333
3,Case,Bpic17DAE-Leaky-150,BPIC17,bpic17-0.3-1,0.544218,0.493827,0.606061
4,Case,Bpic17DAE-Leaky-200,BPIC17,bpic17-0.3-1,0.511111,0.403509,0.696970
5,Case,Bpic17DAE-Leaky-25,BPIC17,bpic17-0.3-1,0.492424,0.492424,0.492424
6,Case,Bpic17DAE-Leaky-250,BPIC17,bpic17-0.3-1,0.546713,0.503185,0.598485
7,Case,Bpic17DAE-Leaky-300,BPIC17,bpic17-0.3-1,0.515901,0.483444,0.553030
8,Case,Bpic17DAE-Leaky-50,BPIC17,bpic17-0.3-1,0.480818,0.362934,0.712121
9,Case,Bpic17LTNFROZEN-10,BPIC17,bpic17-0.3-1,0.521505,0.404167,0.734848


# End

In [84]:
end_time = arrow.now("Europe/Berlin")
print(f"Start time: {start_time}")
print(f"Duration: {end_time - start_time}")
print(f"End time: {end_time}")

Start time: 2025-08-17T10:31:54.714861+02:00
Duration: 0:06:08.184759
End time: 2025-08-17T10:38:02.899620+02:00
